# Monthly Retail CDAO Run

Run this once a month. End-to-end:

1. **SFTP** your monthly CSV from `avalonwinscp.fnb.co.za`
2. **Hash** the PII columns (`cust_id_reg_no`, `EMAIL_ADDR`, `CUST_CELL_NO`)
3. **Convert** CSV → Parquet (chunked, memory-safe)
4. **Upload** the Parquet to `gs://customer_spend_data/`

## What you actually do

Three things, that's it:

1. **Open this notebook** in Jupyter.
2. **Run the cells top to bottom.** Notebook installs packages, runs Google sign-in, asks for your AD credentials, uploads to the test bucket as a tiny smoke test.
3. **Flip VPN off** when the notebook says so (between Cell 7 and Cell 8). That's the one thing it can't do for you.

If the test looks right, change `MODE = 'FULL'` in Cell 2 and run again. Done.

## ⚠ VPN dance (the only manual step)

FNB VPN lets you reach `avalonwinscp` for SFTP, but **blocks GCS**. So:

| Cell | VPN |
|---|---|
| 1–4 (setup, preview) | Either |
| **5 (SFTP)** | **🔵 ON** |
| 6, 7 (local work) | Either |
| **8 (GCS upload)** | **🔴 OFF** |
| 9 (cleanup) | Either |

The notebook reminds you when to flip.

## Two modes — always run TEST first

In Cell 2 you'll see `MODE`. Two settings:

| MODE | What happens |
|---|---|
| `'TEST'` *(start here)* | Processes only the **first 10,000 rows**, uploads to the **test bucket** with a `_TEST_` prefix in the filename. Proves the whole pipeline works without touching prod or wasting time. |
| `'FULL'` | Real run. Whole file, prod bucket. Only switch to this after a TEST that looked right. |

The notebook also **skips re-uploading if the target already exists** in the bucket — so re-running the day after a successful upload is harmless. Set `OVERWRITE = True` only if you really need to force a re-upload.

## Cell 1 — Install packages + GCP auth

Safe to re-run. Handles everything that used to need Terminal: installs Python packages AND runs the one-time `gcloud auth application-default login` if it hasn't been done.

Restart the kernel (Kernel → Restart) after this cell if anything was newly installed.

In [ ]:
%pip install --quiet paramiko pandas pyarrow google-cloud-storage python-dotenv

import subprocess, sys, shutil

print()
print('─' * 60)
print('✓ Python packages installed.')
print('─' * 60)

# Check if gcloud ADC is set up. If not, run the login here so Pierre
# never has to open a Terminal.
have_gcloud = shutil.which('gcloud') is not None
if not have_gcloud:
    print()
    print('⚠ gcloud CLI not found on PATH.')
    print('  Cell 8 (GCS upload) will not work until you install Google Cloud SDK:')
    print('    https://cloud.google.com/sdk/docs/install')
else:
    rc = subprocess.run(
        ['gcloud', 'auth', 'application-default', 'print-access-token'],
        capture_output=True, text=True,
    ).returncode

    if rc == 0:
        print('✓ GCP credentials already set up.')
    else:
        print()
        print('GCP credentials not set up — running login now.')
        print('  A browser will open. Sign in with your FNB Google account.')
        print('  (This is a one-time step.)')
        subprocess.run(['gcloud', 'auth', 'application-default', 'login'])
        print('✓ GCP credentials saved.')

print()
print('─' * 60)
print('→ If any packages were newly installed: Kernel → Restart.')
print('  Otherwise just continue with Cell 2.')

## Cell 2 — Settings

This is the **only cell you normally edit**. Change `STAMP` and the bucket choice; everything else stays the same.

In [ ]:
from datetime import date

# ═══════════════════════════════════════════════════════════════════
#  THE TWO THINGS YOU EDIT
# ═══════════════════════════════════════════════════════════════════

MODE = 'TEST'                # 'TEST' = tiny chunk → test bucket
                             # 'FULL' = real run → prod bucket
                             # Always run TEST first. Flip to FULL only after Cell 8 confirms TEST upload succeeded.

STAMP = date.today().strftime('%Y%m%d')    # e.g. '20260520' — change if you need a specific date

OVERWRITE = False            # set True only if you really need to re-upload an existing file

# ═══════════════════════════════════════════════════════════════════
#  Below this line — leave as-is unless something has changed
# ═══════════════════════════════════════════════════════════════════

STEM         = 'burger'                            # file stem; full name = <STEM>_<STAMP>.csv
TEST_ROWS    = 10_000                              # how many rows the TEST mode processes
SFTP_HOST    = 'avalonwinscp.fnb.co.za'
SFTP_PORT    = 22
REMOTE_DIR   = '/data/fnb/retail_sales_and_cdao/Pierre/'
GCP_PROJECT  = 'fmn-sandbox'
OUT_DIR      = './out'
CHUNKSIZE    = 500_000
COMPRESSION  = 'snappy'
HASH_COLUMNS = ['cust_id_reg_no', 'EMAIL_ADDR', 'CUST_CELL_NO']
INT_COLS     = ['demo_1', 'demo_4', 'demo_8']
FLOAT_COLS   = ['demo_2', 'demo_3', 'demo_7']

assert MODE in ('TEST', 'FULL'), f"MODE must be 'TEST' or 'FULL', got {MODE!r}"

print(f'  MODE:          {MODE}')
print(f'  STAMP:         {STAMP}')
print(f'  Source file:   {STEM}_{STAMP}.csv')
if MODE == 'TEST':
    print(f'  Test rows:     first {TEST_ROWS:,} rows only')
print(f'  Overwrite:     {OVERWRITE}')

## Cell 3 — Credentials + bucket

Reads what it can from a `.env` file (if you have one), prompts for anything missing. No file editing required.

In [ ]:
import os, getpass
from pathlib import Path

# Try to load .env if it exists — otherwise just rely on prompts.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# AD credentials — prompt if missing
AD_USERNAME = os.getenv('AD_USERNAME', '').strip()
AD_PASSWORD = os.getenv('AD_PASSWORD') or os.getenv('DOMAIN_PW') or ''

if not AD_USERNAME:
    AD_USERNAME = input('AD username (e.g. f3799182): ').strip()
if not AD_PASSWORD:
    AD_PASSWORD = getpass.getpass('AD password (hidden as you type): ')

# Buckets — prompt if missing (in TEST mode the test bucket must be known)
PROD_BUCKET = os.getenv('GCP_BUCKET', 'customer_spend_data')
TEST_BUCKET = os.getenv('TEST_BUCKET', '')

if MODE == 'TEST' and not TEST_BUCKET:
    print('TEST_BUCKET not set in .env — asking you here.')
    TEST_BUCKET = input('Name of your TEST bucket (e.g. testing-sandbox-123): ').strip()
    if not TEST_BUCKET:
        raise RuntimeError('No test bucket given. Set TEST_BUCKET in .env or paste it above.')

if MODE == 'TEST':
    BUCKET = TEST_BUCKET
    BUCKET_LABEL = f'TEST: {BUCKET}'
    TARGET_NAME = f'_TEST_{STEM}_{STAMP}.parquet'
else:
    BUCKET = PROD_BUCKET
    BUCKET_LABEL = f'PROD: {BUCKET}  ⚠ this is the real bucket'
    TARGET_NAME = f'{STEM}_{STAMP}.parquet'

print()
print(f'✓ Credentials captured for: {AD_USERNAME}')
print(f'✓ Will upload to: gs://{BUCKET}/{TARGET_NAME}')
print(f'   ({BUCKET_LABEL})')

## Cell 4 — Preview what's about to happen

Just prints what the next cells will do. No network calls — safe regardless of VPN.

The "is it already uploaded?" check happens at upload time (Cell 8), since that's the only point we can talk to GCS.

In [ ]:
file_basename = f'{STEM}_{STAMP}'
remote_path   = REMOTE_DIR.rstrip('/') + '/' + file_basename + '.csv'
out_dir       = Path(OUT_DIR).expanduser()
local_csv     = out_dir / f'{file_basename}.csv'
local_parq    = out_dir / TARGET_NAME

print('=' * 60)
print('  About to run:')
print('=' * 60)
print(f'  Source CSV:   {remote_path}')
if MODE == 'TEST':
    print(f'  Limit:        first {TEST_ROWS:,} rows only')
print(f'  Local CSV:    {local_csv}')
print(f'  Local Parq:   {local_parq}')
print(f'  Hash columns: {HASH_COLUMNS}')
print(f'  Upload to:    gs://{BUCKET}/{TARGET_NAME}')
print('=' * 60)
print()
print('  VPN sequence for this run:')
print('     Cell 5 (SFTP):     VPN ON  🔵')
print('     Cell 6 (parquet):  either, local-only work')
print('     Cell 7 (preview):  either, local-only work')
print('     Cell 8 (upload):   VPN OFF 🔴   ← will check & skip if already uploaded')
print('=' * 60)
print()
print('→ Make sure VPN is ON, then run Cell 5.')

## Cell 5 — SFTP the CSV from avalonwinscp

**🔵 VPN must be ON for this cell.** SFTP only works through the FNB network.

In [ ]:
import time, paramiko

# Quick sanity: skip SFTP if we already have the parquet AND it's the right mode
# (lets Pierre re-run Cell 8 after a VPN-related upload retry without redoing SFTP).
if local_parq.exists() and not OVERWRITE:
    size_mb = local_parq.stat().st_size / 1e6
    print(f'⊘ Skipping SFTP — local parquet already exists:')
    print(f'   {local_parq}  ({size_mb:.1f} MB)')
    print(f'   If you want to re-download from SFTP, delete the local file or set OVERWRITE=True.')
else:
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f'Connecting to {SFTP_HOST}:{SFTP_PORT} as {AD_USERNAME}...')
    print('  (VPN must be ON for this to work.)')
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    try:
        ssh.connect(
            hostname=SFTP_HOST, port=SFTP_PORT,
            username=AD_USERNAME, password=AD_PASSWORD,
            timeout=30, allow_agent=False, look_for_keys=False,
        )
    except (paramiko.SSHException, OSError, TimeoutError) as e:
        print(f'❌ SFTP connect failed: {type(e).__name__}: {e}')
        print()
        print('  Most likely cause: VPN is OFF.')
        print('  Cell 5 needs VPN ON. Connect VPN and re-run this cell.')
        raise

    sftp = ssh.open_sftp()
    try:
        print(f'Downloading {remote_path} ...')
        t0 = time.time()
        sftp.get(remote_path, str(local_csv))
        size_mb = local_csv.stat().st_size / 1e6
        print(f'✓ {size_mb:.1f} MB in {time.time()-t0:.1f}s → {local_csv}')
    finally:
        sftp.close()
        ssh.close()

## Cell 6 — Convert CSV → Parquet (with PII hashing)

In [ ]:
import hashlib, time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

def hash_value(val):
    if pd.isna(val):
        return val
    s = str(val).lower().replace(' ', '')
    return hashlib.sha256(s.encode('utf-8')).hexdigest()

if local_parq.exists() and not OVERWRITE:
    size_mb = local_parq.stat().st_size / 1e6
    print(f'⊘ Skipping conversion — parquet already exists:')
    print(f'   {local_parq}  ({size_mb:.1f} MB)')
else:
    if local_parq.exists():
        local_parq.unlink()

    if not local_csv.exists():
        raise RuntimeError(
            f'No local CSV at {local_csv}. Did Cell 5 run successfully?'
        )

    writer, total, t0 = None, 0, time.time()
    row_limit = TEST_ROWS if MODE == 'TEST' else None

    for chunk in pd.read_csv(
        local_csv,
        chunksize=CHUNKSIZE,
        encoding='ascii',
        engine='python',
        encoding_errors='replace',
        sep=',',
    ):
        if row_limit is not None and total + len(chunk) > row_limit:
            chunk = chunk.iloc[:max(0, row_limit - total)]
            if chunk.empty:
                break

        chunk.columns = chunk.columns.astype(str).str.replace('\ufeff', '', regex=False).str.strip()

        for col in FLOAT_COLS:
            if col in chunk.columns:
                chunk[col] = pd.to_numeric(chunk[col], errors='coerce').astype('float64')
        for col in INT_COLS:
            if col in chunk.columns:
                chunk[col] = pd.to_numeric(chunk[col], errors='coerce').astype('Int64')

        if 'EFF_DATE' in chunk.columns:
            chunk['EFF_DATE'] = pd.to_datetime(chunk['EFF_DATE'], format='%d%b%Y', errors='coerce').dt.date

        for col in HASH_COLUMNS:
            if col in chunk.columns:
                chunk[col] = chunk[col].apply(hash_value)

        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(local_parq, table.schema, compression=COMPRESSION)
        writer.write_table(table)

        total += len(chunk)
        print(f'  ... {total:,} rows', end='\r')

        if row_limit is not None and total >= row_limit:
            break

    if writer is not None:
        writer.close()

    size_mb = local_parq.stat().st_size / 1e6
    mode_note = f'  (TEST mode — capped at {TEST_ROWS:,} rows)' if MODE == 'TEST' else ''
    print(f'\n✓ {total:,} rows → {local_parq.name}  ({size_mb:.1f} MB in {time.time()-t0:.1f}s){mode_note}')

## Cell 7 — Preview the Parquet (sanity check before upload)

Confirms hashing worked and the schema looks right.

In [ ]:
if not local_parq.exists():
    raise RuntimeError(f'No parquet to preview at {local_parq}. Did Cell 6 run?')

preview = pd.read_parquet(local_parq).head(5)
print(f'Columns ({len(preview.columns)}): {list(preview.columns)}')
print()
for col in HASH_COLUMNS:
    if col in preview.columns:
        sample = preview[col].dropna().iloc[0] if preview[col].notna().any() else '(all null)'
        looks_hashed = isinstance(sample, str) and len(sample) == 64
        mark = '✓' if looks_hashed else '⚠'
        print(f'  {mark} {col}: {str(sample)[:80]}')
print()
print('=' * 60)
print('  ⚠  VPN SWITCH REQUIRED BEFORE CELL 8')
print('=' * 60)
print('  SFTP needed VPN ON. GCS upload needs VPN OFF (FNB VPN blocks GCS).')
print()
print('  → DISCONNECT VPN now, then run Cell 8.')
print('=' * 60)
print()
preview

## Cell 8 — Upload to GCS

**🔴 VPN must be OFF for this cell.** FNB VPN blocks GCS traffic — if VPN is still on, this cell will hang or fail.

Uses your `gcloud auth application-default login` credentials. If you haven't run that once on this machine, do so in Terminal first.

In [ ]:
import time
from google.cloud import storage
from google.api_core import exceptions as gexc

print(f'Target: gs://{BUCKET}/{TARGET_NAME}')
print('  (If this cell hangs for >30s, VPN is probably still ON — disconnect it.)')
print()

try:
    client = storage.Client(project=GCP_PROJECT)
    bucket_obj = client.bucket(BUCKET)
    blob = bucket_obj.blob(TARGET_NAME)

    if blob.exists():
        print(f'⚠  gs://{BUCKET}/{TARGET_NAME} ALREADY EXISTS.')
        if OVERWRITE:
            print(f'   OVERWRITE=True in Cell 2 — will replace it.')
            do_upload = True
        else:
            print()
            print(f'   Skipping upload. (Re-running this notebook after a successful upload')
            print(f'   is harmless — you just see this message.)')
            print()
            print(f'   To force a re-upload, set OVERWRITE=True in Cell 2 and re-run Cells 2 → 8.')
            do_upload = False
    else:
        do_upload = True

    if do_upload:
        t0 = time.time()
        blob.upload_from_filename(str(local_parq))
        size_mb = local_parq.stat().st_size / 1e6
        print(f'✓ Uploaded {size_mb:.1f} MB in {time.time()-t0:.1f}s')
        print(f'  URI: gs://{BUCKET}/{TARGET_NAME}')
        print()

        if MODE == 'TEST':
            print('=' * 60)
            print('  ✅ TEST UPLOAD SUCCEEDED')
            print('=' * 60)
            print(f'  Verify in GCS console:  gs://{BUCKET}/{TARGET_NAME}')
            print()
            print('  When you\'re happy, do the FULL run — keep VPN OFF, just:')
            print('    1. In Cell 2, change MODE = \'FULL\'')
            print('    2. Run Cells 2 → 8 again')
            print('       (The CSV is already downloaded; the notebook just processes')
            print('        the WHOLE file this time and uploads to the PROD bucket.)')
            print('=' * 60)
        else:
            print('=' * 60)
            print('  ✅ FULL UPLOAD COMPLETE')
            print('=' * 60)
            print('  Move to Cell 9 to clean up the local CSV.')
            print('=' * 60)

except (gexc.ServiceUnavailable, gexc.DeadlineExceeded, ConnectionError, TimeoutError) as e:
    print(f'❌ GCS connection failed: {type(e).__name__}')
    print()
    print('  Most likely cause: VPN is still ON.')
    print('  FNB VPN blocks GCS traffic. Disconnect VPN and re-run this cell.')
    print()
    print(f'  Details: {e}')
    raise
except gexc.Forbidden as e:
    print(f'❌ Permission denied on the bucket.')
    print(f'   Bucket: gs://{BUCKET}/')
    print(f'   Your account needs Storage Object Admin (or at least objectCreator).')
    print()
    print(f'  Details: {e}')
    raise

## Cell 9 — Tidy up (optional)

Removes the local CSV (which can be large). The parquet is kept in `./out/`.

In [ ]:
KEEP_CSV = False   # set to True if you want to keep the raw CSV

if not KEEP_CSV and local_csv.exists():
    local_csv.unlink()
    print(f'✓ Removed local CSV: {local_csv.name}')
else:
    print(f'  Keeping local CSV: {local_csv}')

print(f'  Parquet retained:  {local_parq}')
print('\n✓ Done.')